# BIRD-dev Execution-Accuracy eval (real databases)

This notebook evaluates **your** fine-tuned model (Llama-3.1-8B QLoRA SFT / GRPO
adapter) on the **official BIRD dev set** with a correct EX pipeline.

What's different from the synthetic eval:
- We query the **real, populated BIRD `.sqlite` files** (rows actually exist), so
  a wrong query and a right query return different results. No empty tables.
- The schema in the prompt is **extracted from the real DB** (true column names),
  not from a hand-written `context` string.
- EX uses the **official BIRD semantics**: `set(fetchall())` over native row
  tuples (so `1 == 1.0`), ORDER BY ignored. Comparable to published BIRD numbers.

### Setup on Kaggle
Add a BIRD dev dataset to the notebook (Add Input → search "BIRD"). You need:
`dev.json` and the `dev_databases/<db_id>/<db_id>.sqlite` files. The config cell
auto-discovers them under `/kaggle/input`. Point `ADAPTER_PATH` at your LoRA
adapter (SFT or GRPO). Set `ADAPTER_PATH = None` to get the base-model baseline.


In [ ]:
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo


In [ ]:
!pip install -q --no-cache-dir trl peft accelerate bitsandbytes sqlparse


In [ ]:
# === CONFIG ===================================================================
import os, glob, json

BASE_MODEL    = "unsloth/Meta-Llama-3.1-8B-Instruct"   # same base you trained on
ADAPTER_PATH  = "/kaggle/working/results/best_model"   # your LoRA adapter; None = base only
MAX_SEQ_LEN   = 4096
MAX_NEW_TOK   = 512
BATCH_SIZE    = 4          # lower to 2 if OOM on long schemas
SAMPLE_ROWS   = 0          # >0 = include N example rows per table in the prompt
EXEC_TIMEOUT  = 30.0       # seconds per query
LIMIT         = 0          # 0 = full dev set; set e.g. 100 for a quick smoke

# ---- auto-discover BIRD dev.json + the databases folder under /kaggle/input ---
def _find(pattern, roots=("/kaggle/input", "/kaggle/working")):
    for root in roots:
        hits = glob.glob(os.path.join(root, "**", pattern), recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    return None

DEV_JSON = _find("dev.json") or _find("mini_dev_sqlite.json")
assert DEV_JSON, "dev.json not found under /kaggle/input — add a BIRD dev dataset."
# any *.sqlite tells us where the DB tree lives
_any_db  = _find("*.sqlite")
assert _any_db, "no *.sqlite found — make sure the BIRD dev_databases are added."
DB_ROOT  = os.path.dirname(os.path.dirname(_any_db))   # .../dev_databases

print("dev.json :", DEV_JSON)
print("db root  :", DB_ROOT)
print("adapter  :", ADAPTER_PATH or "(none — base model baseline)")


In [ ]:
# === LOAD BIRD DEV + RESOLVE EACH DB FILE =====================================
with open(DEV_JSON) as f:
    dev = json.load(f)
if LIMIT:
    dev = dev[:LIMIT]

def db_path_for(db_id):
    # standard BIRD layout: dev_databases/<db_id>/<db_id>.sqlite
    p = os.path.join(DB_ROOT, db_id, f"{db_id}.sqlite")
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(DB_ROOT, "**", f"{db_id}.sqlite"), recursive=True)
    return hits[0] if hits else None

missing = sorted({e["db_id"] for e in dev if not db_path_for(e["db_id"])})
if missing:
    print(f"WARNING: {len(missing)} db files missing, e.g. {missing[:3]}")
print(f"Loaded {len(dev)} BIRD dev examples across "
      f"{len({e['db_id'] for e in dev})} databases")
print("keys:", list(dev[0].keys()))


In [ ]:
# === SCHEMA FROM THE REAL DB (true column names) ==============================
import sqlite3

def schema_ddl(db_path, sample_rows=0):
    """Real CREATE statements (+ optional sample rows) read from the live DB."""
    conn = sqlite3.connect(db_path)
    conn.text_factory = lambda b: b.decode("utf-8", "ignore")
    cur = conn.cursor()
    cur.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql NOT NULL")
    parts = [r[0].strip() + ";" for r in cur.fetchall()]
    if sample_rows:
        cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
        for (t,) in cur.fetchall():
            try:
                cur.execute(f'SELECT * FROM "{t}" LIMIT {sample_rows}')
                cols = [d[0] for d in cur.description]
                rows = cur.fetchall()
                if rows:
                    parts.append(f"/* sample rows from {t}: "
                                 f"{cols} -> {rows} */")
            except Exception:
                pass
    conn.close()
    return "\n".join(parts)

print(schema_ddl(db_path_for(dev[0]["db_id"]))[:600], "...")


In [ ]:
# === LOAD MODEL + ADAPTER (your trained weights) ==============================
import torch, re
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

load_from = ADAPTER_PATH if ADAPTER_PATH else BASE_MODEL
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=load_from, max_seq_length=MAX_SEQ_LEN,
    dtype=None, load_in_4bit=True, device_map={"": 0},
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
FastLanguageModel.for_inference(model)
model.eval()
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

terminators = [tokenizer.eos_token_id]
eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot, int) and eot >= 0:
    terminators.append(eot)

SYSTEM_PROMPT = (
    "You are an expert SQL assistant. Convert the natural language question into a "
    "single valid SQLite query using ONLY the tables and columns in the provided "
    "schema. Return only the SQL query, no explanation."
)

def build_messages(question, schema, evidence=""):
    user = f"Database Schema:\n{schema}\n\n"
    if evidence:
        user += f"External knowledge: {evidence}\n\n"
    user += f"Question: {question}"
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]

def extract_sql(text):
    t = text.strip()
    t = re.sub(r"^```sql\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t); t = re.sub(r"\s*```$", "", t)
    for tok in ["<|eot_id|>", "<|end_of_text|>", "<|begin_of_text|>"]:
        t = t.replace(tok, "")
    return t.strip()

print("trainable check / model ready:", load_from)


In [ ]:
# === BATCHED GENERATION =======================================================
from tqdm.auto import tqdm

@torch.inference_mode()
def generate_batch(messages_list):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False,
               add_generation_prompt=True) for m in messages_list]
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                    max_length=MAX_SEQ_LEN - MAX_NEW_TOK).to(model.device)
    out = model.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                         max_new_tokens=MAX_NEW_TOK, do_sample=False, num_beams=1,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=terminators)
    gen = out[:, enc.input_ids.shape[1]:]
    return [extract_sql(d) for d in tokenizer.batch_decode(gen, skip_special_tokens=True)]

# cache schema per db_id (don't rebuild every example)
schema_cache = {}
def get_schema(db_id):
    if db_id not in schema_cache:
        schema_cache[db_id] = schema_ddl(db_path_for(db_id), SAMPLE_ROWS)
    return schema_cache[db_id]

preds = []
for i in tqdm(range(0, len(dev), BATCH_SIZE), desc="generate"):
    chunk = dev[i:i+BATCH_SIZE]
    msgs = [build_messages(e["question"], get_schema(e["db_id"]),
                           e.get("evidence", "")) for e in chunk]
    preds.extend(generate_batch(msgs))
print("generated", len(preds), "queries")


In [ ]:
# === EXECUTE + SCORE (official BIRD EX semantics) =============================
# EX = set(fetchall()) equality over NATIVE row tuples (1 == 1.0), ORDER BY
# ignored. Gold is executed too; examples whose GOLD fails to run are EXCLUDED
# from the denominator (unreliable gold), exactly like the official scorer.
import threading

def run_sql(db_path, sql, timeout):
    """Execute read-only with a timeout; return (rows_or_None, error_or_None)."""
    result = {}
    def work():
        try:
            conn = sqlite3.connect(db_path)
            conn.text_factory = lambda b: b.decode("utf-8", "ignore")
            cur = conn.cursor()
            cur.execute(sql)
            result["rows"] = cur.fetchall()
            conn.close()
        except Exception as e:
            result["err"] = str(e)
    th = threading.Thread(target=work, daemon=True)
    th.start(); th.join(timeout)
    if th.is_alive():
        return None, "timeout"
    return result.get("rows"), result.get("err")

def ex_match(gold_rows, pred_rows):
    return set(map(tuple, gold_rows)) == set(map(tuple, pred_rows))

n_correct = n_eval = n_gold_fail = n_pred_fail = 0
diff = {}   # difficulty -> [correct, total]
rows_out = []
for e, pred in tqdm(list(zip(dev, preds)), desc="execute"):
    dbp = db_path_for(e["db_id"]); gold = e["SQL"]
    gr, ge = run_sql(dbp, gold, EXEC_TIMEOUT)
    if ge is not None:               # gold itself broken -> exclude
        n_gold_fail += 1
        rows_out.append({**{k:e.get(k) for k in ("question_id","db_id","difficulty","question")},
                         "pred": pred, "gold": gold, "status": "excluded_gold_fail"})
        continue
    n_eval += 1
    pr, pe = run_sql(dbp, pred, EXEC_TIMEOUT)
    if pe is not None:
        n_pred_fail += 1; ok = False; status = "pred_exec_fail"
    else:
        ok = ex_match(gr, pr); status = "match" if ok else "result_mismatch"
    n_correct += int(ok)
    d = e.get("difficulty", "n/a"); diff.setdefault(d, [0,0])
    diff[d][1] += 1; diff[d][0] += int(ok)
    rows_out.append({**{k:e.get(k) for k in ("question_id","db_id","difficulty","question")},
                     "pred": pred, "gold": gold, "correct": ok, "status": status})

EX = n_correct / n_eval if n_eval else 0.0
print("="*50)
print(f"BIRD dev — Execution Accuracy")
print(f"  total examples      : {len(dev)}")
print(f"  excluded (gold fail): {n_gold_fail}")
print(f"  evaluable           : {n_eval}")
print(f"  pred exec failures  : {n_pred_fail}")
print(f"  EX                  : {EX:.4f}  ({n_correct}/{n_eval})")
print("  by difficulty:")
for d,(c,t) in sorted(diff.items()):
    print(f"    {d:10s}: {c/t:.4f}  ({c}/{t})")


In [ ]:
# === SAVE PER-EXAMPLE RESULTS =================================================
import pandas as pd
OUT = "/kaggle/working/bird_dev_eval.jsonl"
with open(OUT, "w") as f:
    f.write(json.dumps({"__summary__": {
        "EX": round(EX,4), "n_eval": n_eval, "n_total": len(dev),
        "excluded_gold_fail": n_gold_fail, "pred_exec_fail": n_pred_fail,
        "adapter": ADAPTER_PATH, "base": BASE_MODEL,
        "by_difficulty": {d: round(c/t,4) for d,(c,t) in diff.items()},
    }}) + "\n")
    for r in rows_out:
        f.write(json.dumps(r) + "\n")
print("wrote", OUT)
pd.DataFrame([r for r in rows_out if "correct" in r]).head()
